# Walkthroughs and Exercises for Deep Learning for Business Made Simple

**Dr. Chester Ismay**

We teach in **Keras 3** running on the **PyTorch** backend. Run the setup cell
first, then work top to bottom. All data is fetched from the web (a GitHub URL
for the hotel data; Hugging Face streaming for the image and text data), so there
is **nothing to upload** -- each dataset loads itself when you run its cell.


In [ ]:
# Run this once per session, then RESTART the runtime if Colab prompts you to.
# Colab already ships torch; we add/upgrade Keras 3 and the datasets library.
!pip install -q -U keras datasets
!pip install -q scikit-learn matplotlib pillow


In [ ]:
# Run this setup cell FIRST. The backend must be chosen BEFORE keras is imported.
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import pandas as pd

# Display all columns and all outputs from each cell
pd.set_option("display.max_columns", None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

keras.utils.set_random_seed(2026)
print("Keras", keras.__version__, "on backend:", keras.backend.backend())

# Intro: Getting Started with Deep Learning for Business

In this course we use **Keras 3** -- a friendly, high-level deep learning API --
running on the **PyTorch** backend. You write simple Keras code; PyTorch does the
heavy lifting underneath. Every walkthrough also includes an **idiomatic PyTorch**
version so you can see what the same model looks like in raw PyTorch.

## Walkthrough: Setting Up Keras (PyTorch backend) and Your First Tensor

### Confirm the environment

In [ ]:
import keras, torch
print("Keras backend:", keras.backend.backend())   # should say 'torch'
print("PyTorch version:", torch.__version__)

> **Common Pitfall:** `os.environ["KERAS_BACKEND"] = "torch"` must run **before**
> `import keras`. If you import keras first and then set it, you'll silently get
> the default backend. When in doubt, restart the kernel and run the setup cell first.

### A first tensor operation

In [ ]:
# A tensor is just an array the framework can run fast math on (and learn from)
x = keras.ops.array([[1.0, 2.0], [3.0, 4.0]])
print("Sum of all elements:", float(keras.ops.sum(x)))
print("Column means:", keras.ops.mean(x, axis=0))

## Exercise: Confirm Your Environment

In [ ]:
# Build a tiny one-line "model" and run data through it -- proves the stack works
layer = keras.layers.Dense(1, activation="sigmoid")
out = layer(keras.ops.array([[0.5, -1.2, 3.0]]))
print("Output shape:", out.shape)
print("If you see a (1, 1) tensor above, you are ready to go.")

> **GenAI tip:** If a setup cell errors, copy the *full* error plus one line of context
> ("I'm using Keras 3 with the PyTorch backend on Colab") into your AI assistant and
> ask for the fix. Re-run to confirm -- never trust a fix you haven't re-run.

# Module 1: Demystifying Neural Networks for Business Insight

We'll predict **hotel booking cancellations** -- a real revenue-management problem.
Every cancellation is a room you could have sold. The dataset has ~119,000 real
bookings.

## Walkthrough 1.1: Predicting Booking Cancellations with a Neural Network

### Load and inspect the data

In [ ]:
url = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/"
       "data/2020/2020-02-11/hotels.csv")
hotels = pd.read_csv(url)
print(hotels.shape)
hotels[["hotel", "lead_time", "adults", "previous_cancellations",
        "adr", "is_canceled"]].head()

In [ ]:
# How often do guests cancel? (our target)
hotels["is_canceled"].value_counts(normalize=True).round(3)

### Prepare the features

In [ ]:
# A handful of numeric drivers a revenue manager would recognize
features = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
            "adults", "previous_cancellations", "booking_changes",
            "total_of_special_requests", "adr"]

data = hotels[features + ["is_canceled"]].dropna().copy()
X = data[features].values.astype("float32")
y = data["is_canceled"].values.astype("float32")

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y)

# Neural networks train far better on standardized inputs. Fit the shift/scale on the
# TRAINING data only, then apply the SAME numbers to validation (see the pitfall below).
mean, std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-8
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
print("Train:", X_train.shape, "Validation:", X_val.shape)

> **Common Pitfall:** Standardize using statistics from the **training** data only,
> then apply the same shift/scale to validation and new data. Computing the mean on
> all the data leaks information about the future into training.

### Build, train, and evaluate the network

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(len(features),)),
    keras.layers.Dense(32, activation="relu"),   # a "team of specialists"
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),  # probability of cancellation
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=10, batch_size=256, verbose=0)
print("Final validation accuracy:",
      round(float(history.history["val_accuracy"][-1]), 3))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("The falling loss IS the model learning")
plt.show()

### Visualize: can the model tell the two groups apart?

In [ ]:
# A model that "separates" the classes pushes cancellations toward 1 and the rest
# toward 0. Overlapping histograms = an indecisive model.
val_probs = model.predict(X_val, verbose=0).ravel()
plt.hist(val_probs[y_val == 0], bins=30, alpha=0.6, label="actually kept")
plt.hist(val_probs[y_val == 1], bins=30, alpha=0.6, label="actually canceled")
plt.axvline(0.5, color="k", ls="--", lw=1, label="default threshold")
plt.xlabel("predicted cancellation probability"); plt.ylabel("number of bookings")
plt.legend(); plt.title("Separation is the model's confidence, made visible")
plt.show()

> **GenAI tip:** Wondering whether ~78% accuracy is actually good? Paste the class
> balance and the validation accuracy into an AI assistant and ask, "is this better than
> always guessing the majority class, and by how much?" -- then check it against the
> baseline we compute in Module 4.

### The same model in idiomatic PyTorch (reference)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert the same train/val arrays into PyTorch tensors (the "_t" = tensor version).
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train).unsqueeze(1)   # shape (N,) -> (N, 1) to match the model output
X_val_t = torch.tensor(X_val)
y_val_t = torch.tensor(y_val).unsqueeze(1)

# Idiomatic PyTorch feeds the model mini-batches via a DataLoader
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

net = nn.Sequential(
    nn.Linear(len(features), 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),               # outputs a raw score ("logit"), not a probability
)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()    # applies the sigmoid for us, more numerically stable

# Train: loop over the data 10 times, one gradient step per mini-batch
for epoch in range(10):
    net.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear last step's gradients
        loss = loss_fn(net(x_batch), y_batch)  # how wrong are we?
        loss.backward()                       # measure how to improve
        optimizer.step()                      # nudge the weights

# Evaluate, one readable step at a time
net.eval()
with torch.no_grad():
    logits = net(X_val_t)                     # raw scores
    probs = torch.sigmoid(logits)             # turn scores into probabilities
    predicted = (probs > 0.5).float()         # 1 if "canceled", else 0
    correct = (predicted == y_val_t).float()  # 1 where the prediction matches the truth
    val_acc = correct.mean().item()           # fraction correct
print("PyTorch validation accuracy:", round(val_acc, 3))

> **Notice:** Keras handled the training loop for you; in raw PyTorch you write
> the `DataLoader` and the `zero_grad -> forward -> loss -> backward -> step` loop
> yourself. Same model, same mini-batching -- just more control and more code.
>
> One difference to call out: the Keras model ended in a `sigmoid`, but the PyTorch
> model outputs a raw score (a "logit"). That's the PyTorch convention -- `BCEWithLogitsLoss`
> folds the sigmoid in for numerical stability, so we apply `sigmoid` ourselves only
> when we want a probability at the end.

## Exercise 1.1: Predicting Average Daily Rate (Regression)

Now the target is a **number** -- the average daily rate (`adr`), i.e. price. The
same network shape works; we just change the last layer and the loss.

In [ ]:
reg_features = hotels[features].copy()
adr_price = hotels["adr"].astype("float32")

# Keep rows with no missing features, a valid price, and a sensible range (drop outliers > $1000)
keep = reg_features.notna().all(axis=1) & adr_price.notna() & (hotels["adr"].between(0, 1000))

X_reg = reg_features[keep].drop(columns=["adr"]).values.astype("float32")
y_reg = adr_price[keep].values

X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=2026)
# Standardize on the training split only, then apply to validation (same rule as before)
reg_mean, reg_std = X_reg_train.mean(0), X_reg_train.std(0) + 1e-8
X_reg_train = (X_reg_train - reg_mean) / reg_std
X_reg_val = (X_reg_val - reg_mean) / reg_std

reg_model = keras.Sequential([
    keras.layers.Input(shape=(X_reg.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1),                    # no activation -> predict any number
])
reg_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
reg_hist = reg_model.fit(X_reg_train, y_reg_train, validation_data=(X_reg_val, y_reg_val),
                         epochs=10, batch_size=256, verbose=0)
print("Validation MAE (dollars off, on average):",
      round(float(reg_hist.history["val_mae"][-1]), 2))

In [ ]:
# Predicted vs actual price: points on the dashed line are perfect predictions
pred_adr = reg_model.predict(X_reg_val, verbose=0).ravel()
plt.scatter(y_reg_val, pred_adr, s=6, alpha=0.25)
lim = [0, 400]
plt.plot(lim, lim, "r--", label="perfect prediction")
plt.xlim(lim); plt.ylim(lim)
plt.xlabel("actual ADR ($)"); plt.ylabel("predicted ADR ($)")
plt.legend(); plt.title("Where the price model is tight -- and where it drifts")
plt.show()

> **Common Pitfall:** For regression, the output layer has **no activation** and the
> loss is `mse` (or `mae`) -- not `sigmoid` + `binary_crossentropy`. Using a sigmoid
> here would squash every prediction between 0 and 1.

## Walkthrough + Exercise 1.2: Use GenAI to Decode Errors and Explain Choices

Models throw confusing errors while you build them, and it isn't always obvious why a
layer is shaped the way it is. Your AI assistant is a fast debugging and explanation
partner -- the skill is prompting clearly and **verifying** (never trust a fix you
haven't re-run).

**A prompt to start from** (paste into your AI assistant, then iterate):

> I'm using Keras 3 on the PyTorch backend. Here is my code and the full error message:
> [paste]. Explain the cause in plain English, give the smallest fix, then explain in one
> sentence each what my hidden Dense layers and the sigmoid output layer are doing.

In [ ]:
# Scratch space for 1.2 -- paste an error or a model summary here and work with your
# AI assistant. (Nothing to run yet; this cell is yours to experiment in.)

> **GenAI tip:** Apply the suggested fix, then **re-run the cell** to confirm it works.
> If the assistant invents a function or argument, check it against the Keras docs.

### Interpretation Questions

1. The classification model reaches ~78-80% accuracy. With ~63% of bookings *not*
   canceled, is that a strong result? (Hint: what would "always predict not-canceled"
   score? We dig into this in Module 4.)
2. The regression model's MAE is in **dollars**. Is being off by that much acceptable
   for a pricing decision? Where would it need to be tighter?
3. Which features would you add to improve either model -- and which might be
   *leakage* (information you wouldn't have at booking time)?

### Self-Check

By the end of this module, you should be able to:

- [ ] Load a business dataset and standardize numeric features for a network
- [ ] Build, compile, and train a feedforward network in Keras
- [ ] Switch a model between **classification** (sigmoid + cross-entropy) and **regression** (no activation + MSE)
- [ ] Read a falling loss curve as evidence of learning
- [ ] Recognize the same model written in idiomatic PyTorch

# Module 2: Deep Learning for Images and Customer Experience

Direct application: **auto-tagging product/food photos** -- the kind of task behind
delivery-app menus, retail catalogs, and visual search. We use the **Food-101**
dataset and a **pre-trained** convolutional network, so we get strong results without
training on millions of images ourselves (transfer learning).

## Walkthrough 2.1: Auto-Tagging Photos with a Pre-Trained CNN

### Grab a few images

In [ ]:
from datasets import load_dataset
from PIL import Image

food = load_dataset("ethz/food101", split="train", streaming=True)
class_names = food.features["label"].names

samples = []
for row in food:
    samples.append(row)
    if len(samples) >= 6:
        break
print("Example categories in this dataset:", class_names[:8], "...")

### Let a pre-trained model label them

In [ ]:
from keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input, decode_predictions)

pretrained = MobileNetV2(weights="imagenet")   # trained on millions of images

def predict_label(pil_img):
    img = pil_img.convert("RGB").resize((224, 224))
    batch = np.array(img, dtype="float32")[None]   # [None] wraps one image in a batch of 1
    arr = preprocess_input(batch)                  # the preprocessing this model expects
    top = decode_predictions(pretrained.predict(arr, verbose=0), top=1)[0][0]
    return top[1], round(float(top[2]), 2)   # (label, confidence)

for row in samples[:4]:
    guess, conf = predict_label(row["image"])
    print(f"true: {class_names[row['label']]:<18} model guess: {guess} ({conf})")

In [ ]:
# Show the images with the model's guess -- seeing the picture makes the result land
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, row in zip(axes, samples[:4]):
    guess, conf = predict_label(row["image"])
    ax.imshow(row["image"]); ax.axis("off")
    ax.set_title(f"true: {class_names[row['label']]}\nguess: {guess} ({conf})", fontsize=9)
plt.tight_layout(); plt.show()

> **Common Pitfall:** Every pre-trained model expects its **own** preprocessing and
> input size (here 224x224 and `preprocess_input`). Skip it and predictions look
> random. When you swap models, swap the matching `preprocess_input`.

> **GenAI tip:** When the pre-trained model guesses a *near-miss* label (say "plate" for
> a pizza), paste its top guesses into an AI assistant and ask why those categories are
> easy to confuse -- a fast way to build intuition for what the model actually "sees."

## Exercise 2.1: Fine-Tune a Classifier for *Your* Categories

The pre-trained model knows ImageNet labels, not *your* product categories. Transfer
learning fixes that: keep the model's general "vision," retrain only a small new head
on your few categories.

In [ ]:
# Build a small labeled set from 3 food categories (your "product catalog")
wanted = ["pizza", "hamburger", "sushi"]
# Map each Food-101 category index to a fresh 0/1/2 label for our 3-class problem
want_idx = {class_names.index(name): new_label for new_label, name in enumerate(wanted)}

imgs, labels = [], []
counts = {category: 0 for category in want_idx}     # how many we've collected per category
for row in load_dataset("ethz/food101", split="train", streaming=True):
    category = row["label"]
    if category in want_idx and counts[category] < 120:   # cap at 120 images each
        imgs.append(np.array(row["image"].convert("RGB").resize((160, 160)), "float32"))
        labels.append(want_idx[category])
        counts[category] += 1
    if all(c >= 120 for c in counts.values()):
        break

X_img = np.stack(imgs)
y_img = np.array(labels)
X_img_train, X_img_val, y_img_train, y_img_val = train_test_split(
    X_img, y_img, test_size=0.25, random_state=2026, stratify=y_img)
print("Training images:", X_img_train.shape, "across", len(wanted), "categories")

In [ ]:
# Freeze the pre-trained base; train only a new classification head
base = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights="imagenet")
base.trainable = False

clf = keras.Sequential([
    keras.layers.Input(shape=(160, 160, 3)),
    keras.layers.Lambda(preprocess_input),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(wanted), activation="softmax"),
])
clf.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
clf.fit(X_img_train, y_img_train, validation_data=(X_img_val, y_img_val),
        epochs=5, batch_size=16, verbose=0)
print("Validation accuracy:", round(float(clf.evaluate(X_img_val, y_img_val, verbose=0)[1]), 3))

In [ ]:
# Show 6 validation photos with the fine-tuned model's call (green = right, red = wrong)
val_pred = clf.predict(X_img_val, verbose=0).argmax(axis=1)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, img, true_i, pred_i in zip(axes, X_img_val[:6], y_img_val[:6], val_pred[:6]):
    ax.imshow(img.astype("uint8")); ax.axis("off")
    ok = (true_i == pred_i)
    ax.set_title(f"{wanted[pred_i]}", color="green" if ok else "red", fontsize=10)
plt.suptitle("Fine-tuned model on YOUR categories"); plt.tight_layout(); plt.show()

> **Common Pitfall:** Keep the pre-trained base **frozen** (`base.trainable = False`)
> when you have little data. Unfreezing everything with a few hundred images usually
> overfits fast and *erases* the valuable general features.

### Idiomatic PyTorch (reference): transfer learning with torchvision

In [ ]:
# Equivalent pattern in raw PyTorch (run in an environment with torchvision):
import torch, torch.nn as nn
from torchvision import models, transforms

weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
net = models.mobilenet_v2(weights=weights)
for p in net.parameters():            # freeze the base
    p.requires_grad = False
net.classifier[1] = nn.Linear(net.last_channel, len(wanted))  # new head

prep = weights.transforms()           # the model's own preprocessing
opt = torch.optim.Adam(net.classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
# ... standard train loop: for each batch -> opt.zero_grad(); loss.backward(); opt.step()

## Walkthrough + Exercise 2.2: Use GenAI to Explain Layers and Suggest Improvements

Use your AI assistant to demystify a model's architecture and weigh changes for your own
use case -- then test, don't just trust.

**A prompt to start from:**

> Here is my Keras model summary: [paste the output of `clf.summary()`]. Explain what each
> layer does in business terms. Then suggest one change to improve accuracy and one to make
> the model smaller or faster, and describe the trade-offs of each.

In [ ]:
# Scratch space for 2.2 -- paste clf.summary() output here, or try a suggested tweak.

> **GenAI tip:** Apply only ONE suggested change at a time and re-evaluate, so you can see
> what actually helped.

### Interpretation Questions

1. The pre-trained model often guesses a *close* ImageNet label (e.g. "plate",
   "pizza") even before fine-tuning. Why is that "general vision" so reusable?
2. With only 120 images per class, fine-tuning still does well. What does that imply
   for a small business that can't label millions of photos?
3. Where would automated image tagging save real time in your organization?

### Self-Check

By the end of this module, you should be able to:

- [ ] Use a pre-trained CNN to classify images with the correct preprocessing
- [ ] Explain transfer learning in business terms ("reuse, don't reinvent")
- [ ] Fine-tune a frozen base on a small set of your own categories
- [ ] Recognize when freezing the base prevents overfitting

# Module 3: Deep Learning for Text and Customer Sentiment

Direct application: **reading customer sentiment at scale** from reviews and support
text. We use real **Yelp business reviews**.

## Walkthrough 3.1: Sentiment from Reviews with Word Embeddings

### Load real reviews

In [ ]:
from datasets import load_dataset

stream = load_dataset("fancyzhx/yelp_polarity", split="train", streaming=True)
texts, sentiments = [], []
for i, row in enumerate(stream):
    texts.append(row["text"]); sentiments.append(row["label"])  # 0 = negative, 1 = positive
    if i >= 5999:
        break
sentiments = np.array(sentiments, dtype="float32")
print("Loaded", len(texts), "reviews. Example:")
print(texts[0][:160], "...")

### Turn words into numbers (a simple tokenizer)

In [ ]:
from collections import Counter

def build_vocab(texts, max_tokens=10000):
    # Count how often each word appears across every review
    counts = Counter(word for text in texts for word in text.lower().split())
    # Give each common word an ID, starting at 2 so 0 stays "padding" and 1 stays "unknown"
    return {word: i + 2 for i, (word, _) in enumerate(counts.most_common(max_tokens - 2))}

def encode(texts, vocab, max_len=150):
    out = np.zeros((len(texts), max_len), dtype="int64")   # rows of zeros = pre-padded
    for i, text in enumerate(texts):
        ids = [vocab.get(word, 1) for word in text.lower().split()][:max_len]  # 1 = unknown word
        out[i, :len(ids)] = ids
    return out

VOCAB_SIZE, MAX_LEN = 10000, 150
vocab = build_vocab(texts, VOCAB_SIZE)
X_text = encode(texts, vocab, MAX_LEN)
X_text_train, X_text_val, y_text_train, y_text_val = train_test_split(
    X_text, sentiments, test_size=0.2, random_state=2026, stratify=sentiments)
print("Encoded shape:", X_text.shape)

> **Common Pitfall:** Keras 3's `TextVectorization` layer needs TensorFlow installed,
> even on the PyTorch backend. A small tokenizer like this keeps the project
> backend-pure -- and shows exactly what "turning text into numbers" means.

### Build an embedding model

In [ ]:
text_model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN,)),
    keras.layers.Embedding(VOCAB_SIZE, 32),        # the "meaning as numbers" map
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
text_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
text_hist = text_model.fit(X_text_train, y_text_train,
                           validation_data=(X_text_val, y_text_val),
                           epochs=5, batch_size=64, verbose=0)
print("Validation accuracy:", round(float(text_hist.history["val_accuracy"][-1]), 3))

### Try it on your own sentences

In [ ]:
def sentiment_of(sentence):
    ids = encode([sentence], vocab, MAX_LEN)
    p = float(text_model.predict(ids, verbose=0)[0][0])
    return f"{'positive' if p > 0.5 else 'negative'} ({p:.2f})"

for s in ["The staff were friendly and the food arrived hot",
          "Waited an hour and the order was completely wrong"]:
    print(sentiment_of(s), "<-", s)

In [ ]:
# A bar chart turns raw probabilities into something you can show a stakeholder
examples = ["The staff were friendly and the food arrived hot",
            "Best brunch in the neighborhood, will be back",
            "Waited an hour and the order was completely wrong",
            "Overpriced and the room was not clean"]
scores = [float(text_model.predict(encode([s], vocab, MAX_LEN), verbose=0)[0][0])
          for s in examples]
colors = ["seagreen" if p > 0.5 else "indianred" for p in scores]
plt.barh(range(len(examples)), scores, color=colors)
plt.yticks(range(len(examples)), [s[:32] + "..." for s in examples], fontsize=8)
plt.axvline(0.5, color="k", ls="--", lw=1)
plt.xlabel("predicted positivity"); plt.xlim(0, 1)
plt.title("Sentiment score per review"); plt.tight_layout(); plt.show()

### Idiomatic PyTorch (reference)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 32)
        self.fc2 = nn.Linear(32, 1)
    def forward(self, x):
        e = self.embed(x).mean(dim=1)          # average the word vectors
        return self.fc2(torch.relu(self.fc1(e)))  # logits

sentiment_net = SentimentNet(VOCAB_SIZE)
optimizer = torch.optim.Adam(sentiment_net.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

# Mini-batches, just like Keras uses
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_text_train), torch.tensor(y_text_train).unsqueeze(1)),
    batch_size=64, shuffle=True)

# Same training loop shape as Module 1
for epoch in range(5):
    sentiment_net.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(sentiment_net(x_batch), y_batch)
        loss.backward()
        optimizer.step()

# Evaluate, one readable step at a time
sentiment_net.eval()
with torch.no_grad():
    logits = sentiment_net(torch.tensor(X_text_val))
    probs = torch.sigmoid(logits)
    predicted = (probs > 0.5).float()
    correct = (predicted == torch.tensor(y_text_val).unsqueeze(1)).float()
    acc = correct.mean().item()
print("PyTorch validation accuracy:", round(acc, 3))

## Exercise 3.1: Improve and Stress-Test the Model

In [ ]:
# Idea 1: more vocabulary and a longer window often help -- try it
BIG_VOCAB_SIZE, BIG_MAX_LEN = 15000, 200
big_vocab = build_vocab(texts, BIG_VOCAB_SIZE)
X_big = encode(texts, big_vocab, BIG_MAX_LEN)
X_big_train, X_big_val, y_big_train, y_big_val = train_test_split(
    X_big, sentiments, test_size=0.2, random_state=2026, stratify=sentiments)

big_model = keras.Sequential([
    keras.layers.Input(shape=(BIG_MAX_LEN,)),
    keras.layers.Embedding(BIG_VOCAB_SIZE, 32),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
big_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
big_hist = big_model.fit(X_big_train, y_big_train,
                         validation_data=(X_big_val, y_big_val),
                         epochs=5, batch_size=64, verbose=0)
print("Bigger-vocab validation accuracy:", round(float(big_hist.history["val_accuracy"][-1]), 3))

In [ ]:
# Stress-test: feed deliberately tricky reviews (sarcasm, negation) and watch where a
# word-averaging model slips -- it has no sense of word ORDER, so "not great" looks
# positive (it saw "great"). Add your own lines and see which ones fool it.
tricky = [
    "Not great, would not recommend",
    "Well that was a complete waste of money",
    "The food was anything but fresh",
    "I can't say I loved it",
]
for s in tricky:
    print(sentiment_of(s), "<-", s)

> **GenAI tip:** Paste 3-4 reviews the model gets *wrong* into an AI assistant and ask
> "why might a simple averaging model miss the sentiment here?" -- great way to surface
> sarcasm, negation, and context limits to discuss with the room.

## Walkthrough + Exercise 3.2: Use GenAI to Summarize Sentiment for Stakeholders

The value of a sentiment model is the decision it informs. Use your AI assistant to turn
raw numbers into a short narrative two different audiences can act on.

**A prompt to start from:**

> Here are my sentiment results: [paste the validation accuracy and 3-4 example reviews
> with their predicted scores]. Write a 3-sentence summary for a non-technical manager --
> what we found, why it matters, and one caution. Then write a more technical version for a
> data analyst.

In [ ]:
# Scratch space for 3.2 -- assemble the numbers/examples you'll paste into your assistant.

> **GenAI tip:** Fact-check every number the assistant repeats before it goes in a deck --
> it will sometimes "round" or invent figures.

### Interpretation Questions

1. Our model **averages** word vectors. What kinds of reviews will that miss
   (think "not great", sarcasm, "would not recommend")?
2. Where would automated sentiment change a decision your team makes today?
3. When is a pre-trained LLM (just prompt it) the better tool than training this model?

### Self-Check

By the end of this module, you should be able to:

- [ ] Turn raw text into padded integer sequences a model can use
- [ ] Build and train an embedding-based sentiment model in Keras
- [ ] Score new sentences and spot where a simple model struggles
- [ ] Read the same model written as a PyTorch `nn.Module`

# Module 4: Interpreting and Communicating Deep Learning Results

A model is only as valuable as the trust and clarity you can put around it. We
evaluate the **cancellation model** from Module 1 the way a business actually should.

> **Heads up:** this module reuses the trained `model` and the `X_val` / `y_val` split
> from Module 1, so run those cells first (in the same session) before this one.

## Walkthrough 4.1: Confusion Matrix and Business Metrics

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

probs = model.predict(X_val, verbose=0).ravel()
preds = (probs > 0.5).astype(int)

# The bar to beat: always guessing the majority class ("not canceled")
baseline = 1 - y_val.mean()
print(f"Majority-class baseline (always 'not canceled'): {baseline:.1%} accuracy")

cm = confusion_matrix(y_val, preds)
ConfusionMatrixDisplay(cm, display_labels=["not canceled", "canceled"]).plot(
    cmap="Blues", colorbar=False)
plt.title("Where the model is right -- and wrong"); plt.show()

print("Precision:", round(precision_score(y_val, preds), 3),
      "| Recall:", round(recall_score(y_val, preds), 3),
      "| F1:", round(f1_score(y_val, preds), 3))

> **Common Pitfall:** Accuracy alone hides the costly errors. Read the confusion
> matrix by **business cost**: a missed cancellation (false negative) is a room you
> didn't resell; a false alarm (false positive) might mean wrongly hassling a guest.

## Exercise 4.1: Tune the Threshold to the Business Trade-off

The model outputs a *probability*. **You** decide where to draw the line -- and that
choice is a business decision, not a default of 0.5.

In [ ]:
print(f"{'threshold':>10} {'precision':>10} {'recall':>8}")
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (probs > thr).astype(int)
    print(f"{thr:>10} {precision_score(y_val, p):>10.3f} {recall_score(y_val, p):>8.3f}")

In [ ]:
# The trade-off, drawn: as the line moves left, you catch more but cry wolf more
grid = np.linspace(0.1, 0.9, 33)
prec = [precision_score(y_val, (probs > t).astype(int), zero_division=0) for t in grid]
rec = [recall_score(y_val, (probs > t).astype(int)) for t in grid]
plt.plot(grid, prec, label="precision")
plt.plot(grid, rec, label="recall")
plt.axvline(0.35, color="k", ls="--", lw=1, label="a recall-leaning choice")
plt.xlabel("decision threshold"); plt.ylabel("score"); plt.legend()
plt.title("Precision vs recall is a dial you set to the business cost")
plt.show()

In [ ]:
# Suppose missing a cancellation costs 4x a false alarm -> favor recall
chosen = 0.35
p = (probs > chosen).astype(int)
print(f"At threshold {chosen}: we catch {recall_score(y_val, p):.0%} of cancellations, "
      f"with precision {precision_score(y_val, p):.0%}.")

> **GenAI tip:** Feed these numbers to an AI assistant and ask for a 3-sentence
> executive summary: the decision, the impact, and the main risk. Then fact-check
> every number it repeats before it goes in a deck.

## Walkthrough + Exercise 4.2: Use GenAI to Draft an Executive Summary

Turn your evaluation into a decision-ready brief. The assistant drafts; you verify and own
the recommendation.

**A prompt to start from:**

> Given these results -- precision [X] and recall [Y] at a decision threshold of [T], where
> missing a cancellation costs about 4x a false alarm -- write a 3-sentence executive
> summary: the decision, the business impact, and the main risk. Keep it jargon-free.

In [ ]:
# Scratch space for 4.2 -- paste your chosen threshold's precision/recall here.

> **GenAI tip:** Check the draft against what the model actually showed -- the numbers, and
> whether the recommendation matches your stated business cost.

### Interpretation Questions

1. As you lower the threshold, recall rises but precision falls. Which way should a
   revenue team lean for cancellations -- and why?
2. Write one sentence a non-technical manager would understand describing what this
   model does and what it's worth.
3. What would you monitor after deployment to catch the model going stale (drift)?

### Self-Check

By the end of this module, you should be able to:

- [ ] Build and read a confusion matrix in business terms
- [ ] Compute precision, recall, and F1 -- and say when accuracy misleads
- [ ] Tune a decision threshold to a stated business cost
- [ ] Turn model results into a clear, honest stakeholder summary